In [38]:
import pandas as pd
import numpy as np
import joblib
from sqlalchemy import create_engine

In [39]:
import os
os.environ["DB_PASSWORD"] = "2602"

In [40]:
import os

def get_engine():
    db_user = os.environ.get("DB_USER", "postgres")
    db_password = os.environ["DB_PASSWORD"]
    db_host = os.environ.get("DB_HOST", "::1")
    db_port = os.environ.get("DB_PORT", "5432")
    db_name = os.environ.get("DB_NAME", "telco_churn")
    return create_engine(
        f"postgresql+psycopg2://{db_user}:{db_password}@[{db_host}]:{db_port}/{db_name}"
    )

engine = get_engine()

try:
    with engine.connect():
        print("PostgreSQL connected successfully!")
except Exception as e:
    print("Connection failed:")
    print(e)
    raise

PostgreSQL connected successfully!


In [41]:
query = """
SELECT *
FROM public.customers
"""

raw_df = pd.read_sql(query, engine)

print("\nData loaded successfully")
print("Shape:", raw_df.shape)

if raw_df.shape[1] != 23:
    print("WARNING: Expected 23 columns")


Data loaded successfully
Shape: (7043, 23)


In [42]:
CANONICAL_COLUMNS = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "Churn", "TenureGroup", "TotalServices"
]

def normalize_columns(raw_df):
    raw_df = raw_df.rename(columns=lambda c: c.strip())
    lower_to_actual = {c.lower(): c for c in raw_df.columns}

    rename_map = {}
    missing = []
    for col in CANONICAL_COLUMNS:
        key = col.lower()
        if key in lower_to_actual:
            rename_map[lower_to_actual[key]] = col
        else:
            missing.append(col)

    if missing:
        raise ValueError(
            f"Missing columns (checked case-insensitively): {missing}\n"
            f"Actual columns found: {raw_df.columns.tolist()}"
        )

    return raw_df.rename(columns=rename_map)

raw_df = normalize_columns(raw_df)
print("All required columns are present (case-insensitive match).")

All required columns are present (case-insensitive match).


In [43]:
# Postgres folds unquoted identifiers to lowercase, so map back to the
# CamelCase names the rest of this script expects.
column_rename_map = {
    "customerid": "customerID",
    "seniorcitizen": "SeniorCitizen",
    "partner": "Partner",
    "dependents": "Dependents",
    "phoneservice": "PhoneService",
    "multiplelines": "MultipleLines",
    "internetservice": "InternetService",
    "onlinesecurity": "OnlineSecurity",
    "onlinebackup": "OnlineBackup",
    "deviceprotection": "DeviceProtection",
    "techsupport": "TechSupport",
    "streamingtv": "StreamingTV",
    "streamingmovies": "StreamingMovies",
    "contract": "Contract",
    "paperlessbilling": "PaperlessBilling",
    "paymentmethod": "PaymentMethod",
    "monthlycharges": "MonthlyCharges",
    "totalcharges": "TotalCharges",
    "churn": "Churn",
    "tenuregroup": "TenureGroup",
    "totalservices": "TotalServices",
    "gender": "gender",
    "tenure": "tenure",
}

raw_df = raw_df.rename(columns=column_rename_map)

In [44]:
required_columns = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "Churn", "TenureGroup", "TotalServices"
]

missing_columns = [
    col for col in required_columns
    if col not in raw_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in PostgreSQL table: {missing_columns}"
    )

print("All required columns are present.")

All required columns are present.


In [45]:
data = raw_df.drop(
    columns=["customerID", "Churn"]
).copy()

print("\nInitial feature shape:", data.shape)


Initial feature shape: (7043, 21)


In [46]:
data["gender"] = data["gender"].map({
    "Male": 1,
    "Female": 0
})

In [47]:
binary_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in binary_columns:
    data[col] = data[col].map({
        "Yes": 1,
        "No": 0
    })

In [48]:
service_columns = [
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in service_columns:

    # Collapse special categories
    data[col] = data[col].replace({
        "No internet service": "No",
        "No phone service": "No"
    })

    # Encode
    data[col] = data[col].map({
        "Yes": 1,
        "No": 0
    })

In [49]:
tenure_mapping = {
    "0-1 Year": 0,
    "1-2 Years": 1,
    "2-4 Years": 2,
    "4-6 Years": 3
}

data["TenureGroup"] = data["TenureGroup"].map(
    tenure_mapping
)

In [50]:
numeric_charge_columns = ["MonthlyCharges", "TotalCharges"]

for col in numeric_charge_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

charge_nulls = data[numeric_charge_columns].isnull().sum()
charge_nulls = charge_nulls[charge_nulls > 0]

if len(charge_nulls) > 0:
    print("\nFilling blank/invalid numeric charge values with 0:")
    print(charge_nulls)

# tenure = 0 customers legitimately have 0 total charges so far
data[numeric_charge_columns] = data[numeric_charge_columns].fillna(0)

In [51]:
one_hot_columns = [
    "InternetService",
    "Contract",
    "PaymentMethod"
]

data = pd.get_dummies(
    data,
    columns=one_hot_columns,
    drop_first=True
)

In [52]:
bool_columns = data.select_dtypes(
    include="bool"
).columns

data[bool_columns] = data[bool_columns].astype(int)

In [53]:
print("\nChecking missing values...")

missing_values = data.isnull().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) > 0:
    print("WARNING: Missing values found:")
    print(missing_values)
    raise ValueError(
        "Preprocessing produced missing values. "
        "Check the category mappings."
    )
else:
    print("No missing values found.")


Checking missing values...
No missing values found.


In [54]:
non_numeric_columns = data.select_dtypes(
    exclude=np.number
).columns.tolist()

if non_numeric_columns:
    raise ValueError(
        f"Non-numeric columns remain: {non_numeric_columns}"
    )

print("All features are numeric.")

All features are numeric.


In [55]:
infinite_values = np.isinf(
    data.to_numpy(dtype=float)
).sum()

print("Infinite values:", infinite_values)

if infinite_values > 0:
    raise ValueError(
        "Infinite values detected in processed data."
    )

Infinite values: 0


In [56]:
feature_order = data.columns.tolist()

print("\n======================================")
print("FINAL PREPROCESSED DATA")
print("======================================")
print("Rows:", data.shape[0])
print("Features:", data.shape[1])
print("\nFeature order:")

for i, feature in enumerate(feature_order, start=1):
    print(f"{i:02d}. {feature}")


FINAL PREPROCESSED DATA
Rows: 7043
Features: 25

Feature order:
01. gender
02. SeniorCitizen
03. Partner
04. Dependents
05. tenure
06. PhoneService
07. MultipleLines
08. OnlineSecurity
09. OnlineBackup
10. DeviceProtection
11. TechSupport
12. StreamingTV
13. StreamingMovies
14. PaperlessBilling
15. MonthlyCharges
16. TotalCharges
17. TenureGroup
18. TotalServices
19. InternetService_Fiber optic
20. InternetService_No
21. Contract_One year
22. Contract_Two year
23. PaymentMethod_Credit card (automatic)
24. PaymentMethod_Electronic check
25. PaymentMethod_Mailed check


In [57]:
preprocessing_package = {
    "feature_order": feature_order,
    "drop_columns": ["customerID", "Churn"],
    "gender_mapping": {"Male": 1, "Female": 0},
    "binary_columns": binary_columns,
    "binary_mapping": {"Yes": 1, "No": 0},
    "service_columns": service_columns,
    "service_collapse_mapping": {
        "No internet service": "No",
        "No phone service": "No"
    },
    "service_mapping": {"Yes": 1, "No": 0},
    "tenure_mapping": tenure_mapping,
    "numeric_charge_columns": numeric_charge_columns,
    "numeric_charge_fill_value": 0,
    "one_hot_columns": one_hot_columns,
    "drop_first": True
}

In [58]:
package_path = "preprocessing_package.pkl"

joblib.dump(
    preprocessing_package,
    package_path
)

print("\n======================================")
print("PREPROCESSING PACKAGE SAVED")
print("======================================")
print("File:", package_path)


PREPROCESSING PACKAGE SAVED
File: preprocessing_package.pkl


In [59]:
loaded_package = joblib.load(package_path)

print("\nPackage loaded successfully.")
print("Stored feature count:", len(loaded_package["feature_order"]))
print("\nStored feature order:")

for i, feature in enumerate(loaded_package["feature_order"], start=1):
    print(f"{i:02d}. {feature}")


Package loaded successfully.
Stored feature count: 25

Stored feature order:
01. gender
02. SeniorCitizen
03. Partner
04. Dependents
05. tenure
06. PhoneService
07. MultipleLines
08. OnlineSecurity
09. OnlineBackup
10. DeviceProtection
11. TechSupport
12. StreamingTV
13. StreamingMovies
14. PaperlessBilling
15. MonthlyCharges
16. TotalCharges
17. TenureGroup
18. TotalServices
19. InternetService_Fiber optic
20. InternetService_No
21. Contract_One year
22. Contract_Two year
23. PaymentMethod_Credit card (automatic)
24. PaymentMethod_Electronic check
25. PaymentMethod_Mailed check


In [60]:
print("\n======================================")
print("PREPROCESSING COMPLETE")
print("======================================")
print("PostgreSQL rows       :", len(raw_df))
print("Original columns      :", len(raw_df.columns))
print("Processed features    :", len(feature_order))
print("Package               :", package_path)
print("Status                : READY")


PREPROCESSING COMPLETE
PostgreSQL rows       : 7043
Original columns      : 23
Processed features    : 25
Package               : preprocessing_package.pkl
Status                : READY
